## Text input

https://platform.openai.com/docs/models

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import create_agent

# Note - if using ollama, this would need to change
# See Model section in Agents documentation: https://docs.langchain.com/oss/python/langchain/agents?_gl=1*1bs1tlz*_gcl_aw*R0NMLjE3OTA5NzU3NjYuRUFJYUlRb2JDaE1JanZ6ZjBLQ2Nsd01WYnBaUUJoMkhUUTNjRUFBWUFTQUFFZ0xaYWZEX0J3RQ..*_gcl_au*MTkzODc2NzYxLjE3ODk2Njk0MzU.*_ga*MTM4OTM4ODg5Ny4xNzg5NjY5NDM1*_ga_47WX3HKKY2*czE3OTA5NzU3NjYkbzIkZzEkdDE3OTA5Nzc5NTMkajU0JGwwJGgw#model
# The format would be '<provider>:<model>', so if I wanted to use gemma4, I would expect it to be 'ollama:gemma4'
# The provider may not be necessary it seems, but it is clearer to include it.
agent = create_agent(
    model='ollama:gemma4',
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [3]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

***The Capital City of The Moon: Selene Prime***

Selene Prime is not merely a city; it is a self-sustaining biomechanical triumph—a testament to humanity's resilience and ambition etched onto the sterile, majestic canvas of the lunar surface. It stands as the governing heart, research hub, and economic core of the nascent lunar civilization.

---

### 🌕 Overview and Location

**Name:** Selene Prime (Derived from *Selene*, the Greek goddess of the Moon, and *Prime*, denoting superior status/origin).

**Location:** The city is strategically located within the vast, stable basin of the **Mare Tranquillitatis** (Sea of Tranquility), but it does not sit *on* the surface. Instead, it is a complex, tiered mega-structure built *into* a series of deep, sheltered lava tubes and excavated caldera remnants. This provides natural radiation shielding and stable geothermal access.

**Operational Principle:** Selene Prime operates under a closed-loop, bio-regenerative system. It is a monument to effi

## Image input

In [4]:
from ipywidgets import FileUpload
from IPython.display import display
 
uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [5]:
print(uploader.value)

({'name': 'moon.png', 'type': 'image/png', 'size': 358916, 'content': <memory at 0x10f3b3e80>, 'last_modified': datetime.datetime(2026, 10, 2, 21, 43, 47, 954000, tzinfo=datetime.timezone.utc)},)


In [6]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [7]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

This city is not merely built *in* a canyon; it is grown *from* the necessity of the canyon itself. It is a breathtaking monument to human (or post-human) ingenuity, surviving at the intersection of raw planetary power and impossible technology.

I call it **Aethelgard.**

***

### 🌐 Aethelgard: The Citadel of the Ascended

**Location:** Nestled within a colossal, deeply eroded mineral basin on a world designated "Exilon," a planet known for its extreme and variable climate cycles.

**Motto:** *Verticality is Survival.*

**Overview:**
Aethelgard is a mega-structure defined by its sheer verticality. The city doesn't sprawl; it ascends. The immense spires piercing the low, cloudy sky are not simply buildings—they are hyper-optimized life-support stacks, resource processors, and data conduits, all woven together like a vast, metallic coral reef.

The city masters the difficult terrain by treating the rugged, dusty canyons as both its greatest threat and its primary structural support. The

## Audio input

In [8]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:05<00:00,  9.54it/s]


Done.


In [10]:
# agent = create_agent(
#     model='gpt-audio',
# )

agent = create_agent(
    model='ollama:gemma4',
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

ValueError: Blocks of type audio not supported.